In [ ]:

# Standard Library
import os
import re
import json
import time
import uuid
import sqlite3
import hashlib
import logging
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict
from datetime import datetime

# Environment Variables
from dotenv import load_dotenv

# Numerical Computing
import numpy as np

# Data Processing
import pandas as pd

# NLP
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

# Embeddings
from sentence_transformers import SentenceTransformer

# Vector Search
import faiss

# Knowledge Graph
import networkx as nx

# LLM Client (OpenAI SDK)
from openai import OpenAI

# Retry Logic
from tenacity import retry, stop_after_attempt, wait_exponential

# Progress Bars
from tqdm import tqdm


from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List
import os
from dotenv import load_dotenv

# CONFIGURATION

load_dotenv()


@dataclass
class Config:
    NVIDIA_API_KEY: str = os.getenv("NVIDIA_API_KEY", "")
    BASE_URL: str = "https://integrate.api.nvidia.com/v1"

    LLM_MODEL: str = "meta/llama-3.3-70b-instruct"
    EMBEDDING_MODEL: str = "sentence-transformers/all-MiniLM-L6-v2"
    SPACY_MODEL: str = "en_core_web_sm"

    DATABASE_PATH: Path = Path("memory_ai.db")
    FAISS_INDEX_PATH: Path = Path("memory_index.faiss")
    GRAPH_PATH: Path = Path("memory_graph.pkl")
    LOG_PATH: Path = Path("memory.log")

    EMBEDDING_DIMENSION: int = 384

    MAX_NOTE_LENGTH: int = 10000
    MIN_NOTE_LENGTH: int = 20

    SUMMARY_MAX_WORDS: int = 120
    MAX_KEYWORDS: int = 15
    MAX_ENTITIES: int = 50
    MAX_RELATIONSHIPS: int = 50

    IMPORTANCE_THRESHOLD: float = 0.50
    DUPLICATE_THRESHOLD: float = 0.95
    SEARCH_SIMILARITY_THRESHOLD: float = 0.60

    TOP_K_VECTOR_RESULTS: int = 10
    TOP_K_KEYWORD_RESULTS: int = 10
    TOP_K_GRAPH_RESULTS: int = 10

    TEMPERATURE: float = 0.2
    MAX_TOKENS: int = 2048
    RETRY_ATTEMPTS: int = 3
    REQUEST_TIMEOUT: int = 60

    PROMPTS: Dict[str, str] = field(default_factory=lambda: {
        "summary": "Summarize the note while preserving all important facts.",
        "memory_extraction": "Extract structured memories as valid JSON.",
        "consolidation": "Merge related memories into higher-level memories.",
        "analysis": "Analyze memories for goals, skills, beliefs, interests, habits, personality, contradictions and predictions.",
        "qa": "Answer only using the retrieved memories. If unsupported, explicitly say so."
    })


config = Config()


def load_config():
    return config


def get_prompt(name: str):
    return config.PROMPTS[name]


#  DATA LAYER


DB = str(config.DATABASE_PATH)


def get_connection():
    conn = sqlite3.connect(DB)
    conn.execute("PRAGMA foreign_keys = ON")
    conn.row_factory = sqlite3.Row
    return conn


def initialize_database():
    tables = {
        "notes": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            uuid TEXT UNIQUE,
            raw_text TEXT,
            clean_text TEXT,
            summary TEXT,
            importance REAL,
            created_at TEXT
        """,
        "memories": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            note_id INTEGER,
            memory_type TEXT,
            content TEXT,
            confidence REAL,
            FOREIGN KEY(note_id) REFERENCES notes(id)
        """,
        "topics": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            note_id INTEGER,
            topic TEXT,
            score REAL,
            FOREIGN KEY(note_id) REFERENCES notes(id)
        """,
        "entities": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            note_id INTEGER,
            entity TEXT,
            label TEXT,
            FOREIGN KEY(note_id) REFERENCES notes(id)
        """,
        "relationships": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            source_entity TEXT,
            relationship TEXT,
            target_entity TEXT,
            confidence REAL
        """,
        "embeddings": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            note_id INTEGER,
            vector BLOB,
            dimension INTEGER,
            FOREIGN KEY(note_id) REFERENCES notes(id)
        """,
        "observations": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            description TEXT,
            confidence REAL
        """,
        "predictions": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            prediction TEXT,
            probability REAL,
            created_at TEXT
        """,
        "metadata": """
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            note_id INTEGER,
            key TEXT,
            value TEXT,
            FOREIGN KEY(note_id) REFERENCES notes(id)
        """
    }

    conn = get_connection()
    cur = conn.cursor()

    for table, schema in tables.items():
        cur.execute(f"CREATE TABLE IF NOT EXISTS {table} ({schema})")

    conn.commit()
    conn.close()


def save_record(table: str, data: dict):
    conn = get_connection()
    cur = conn.cursor()

    cols = ",".join(data.keys())
    vals = ",".join("?" * len(data))

    cur.execute(
        f"INSERT INTO {table} ({cols}) VALUES ({vals})",
        tuple(data.values())
    )

    conn.commit()
    rowid = cur.lastrowid
    conn.close()
    return rowid


def load_records(table: str, where: str = "", params=()):
    conn = get_connection()
    cur = conn.cursor()

    query = f"SELECT * FROM {table}"
    if where:
        query += " WHERE " + where

    cur.execute(query, params)

    rows = [dict(r) for r in cur.fetchall()]
    conn.close()
    return rows


def update_record(table: str, data: dict, where: str, params=()):
    conn = get_connection()
    cur = conn.cursor()

    fields = ",".join(f"{k}=?" for k in data.keys())

    cur.execute(
        f"UPDATE {table} SET {fields} WHERE {where}",
        tuple(data.values()) + tuple(params)
    )

    conn.commit()
    conn.close()


def delete_record(table: str, where: str, params=()):
    conn = get_connection()
    cur = conn.cursor()

    cur.execute(
        f"DELETE FROM {table} WHERE {where}",
        params
    )

    conn.commit()
    conn.close()


def search_notes(keyword: str):
    return load_records(
        "notes",
        "clean_text LIKE ? OR summary LIKE ?",
        (f"%{keyword}%", f"%{keyword}%")
    )


#  LLM CLIENT


client = OpenAI(
    base_url=config.BASE_URL,
    api_key=config.NVIDIA_API_KEY
)


@retry(
    stop=stop_after_attempt(config.RETRY_ATTEMPTS),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    reraise=True
)
def call_llm(prompt: str, schema: dict | None = None) -> Any:
    """
    Central entry point for every LLM request.

    Parameters
    ----------
    prompt : User prompt.
    schema : Optional JSON schema. If supplied, attempts to parse JSON.

    Returns
    -------
    str | dict
    """

    messages = [
        {
            "role": "system",
            "content": "You are a structured memory extraction assistant."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = client.chat.completions.create(
        model=config.LLM_MODEL,
        messages=messages,
        temperature=config.TEMPERATURE,
        max_tokens=config.MAX_TOKENS,
        timeout=config.REQUEST_TIMEOUT,
    )

    text = response.choices[0].message.content.strip()

    if schema is None:
        return text

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        raise ValueError("LLM did not return valid JSON.")